In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [4]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']

df = df[base]

# Q1

In [5]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

# Q2

In [6]:
df.horsepower.mean()

np.float64(254.44601556505535)

# Q3

In [7]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [8]:
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

In [9]:
def prepare_X(df, filler):
    df_num = df[features]

    df_num = df_num.fillna(filler)
    X = df_num.values
    
    return X

In [10]:
def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

In [11]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X= np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [12]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

In [13]:
mean_hp = df_train.horsepower.mean()

In [14]:
for name, fill in [('with 0', 0), ('with mean', mean_hp)]:
    X_train = prepare_X(df_train, fill)
    w0, w = train_linear_regression(X_train, y_train)

    X_val = prepare_X(df_val, fill)
    y_pred = w0 + X_val.dot(w)

    print(name, round(rmse(y_val, y_pred), 3))

with 0 2.205
with mean 2.202


# Q4

In [15]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X= np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [16]:
def prepare_X(df):
    df_num = df[features]

    df_num = df_num.fillna(0)
    X = df_num.values
    
    return X

In [17]:
scores = {}
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train)
    w0, w = train_linear_regression_reg(X_train, y_train, r)
    
    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)
    scores[r] = round(rmse(y_val, y_pred), 4)
    
    print(r, '=', scores[r])

0 = 2.2053
0.01 = 2.2058
0.1 = 2.2241
1 = 2.3492
5 = 2.4094
10 = 2.4195
100 = 2.4292


# Q5

In [18]:
rmse_list = {}
for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values

    X_train = prepare_X(df_train)
    w0, w = train_linear_regression(X_train, y_train)
    
    X_val = prepare_X(df_val)
    y_pred = w0 + X_val.dot(w)

    rmse_list[seed] = rmse(y_val, y_pred)

    print (seed, '=', rmse_list)

0 = {0: np.float64(2.2392041430461465)}
1 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907)}
2 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907), 2: np.float64(2.163419778753095)}
3 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907), 2: np.float64(2.163419778753095), 3: np.float64(2.2043588768539144)}
4 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907), 2: np.float64(2.163419778753095), 3: np.float64(2.2043588768539144), 4: np.float64(2.2018800943312926)}
5 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907), 2: np.float64(2.163419778753095), 3: np.float64(2.2043588768539144), 4: np.float64(2.2018800943312926), 5: np.float64(2.2457851509084974)}
6 = {0: np.float64(2.2392041430461465), 1: np.float64(2.2021128210838907), 2: np.float64(2.163419778753095), 3: np.float64(2.2043588768539144), 4: np.float64(2.2018800943312926), 5: np.float64(2.2457851509084974), 6: np.float64(2.269721207

In [19]:
def prepare_X(df):
    df_num = df[features]

    df_num = df_num.fillna(0)
    X = df_num.values
    
    return X

In [20]:
np.std(list(rmse_list.values()))

np.float64(0.028781274078187116)

# Q6

In [21]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [22]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

In [23]:
df_full_train = pd.concat([df_train, df_val])

In [24]:
y_full_train = np.concatenate([y_train, y_val])

In [26]:
X_full_train = prepare_X(df_full_train)
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r = 0.001)

y_test = df_test.fuel_efficiency_mpg.values

X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)

round(rmse(y_test, y_pred), 3)

np.float64(2.236)